# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading, exploring, and processing the FAIR^2 dataset using the `mlcroissant` library. It applies the Croissant schema to dynamically explore record sets and fields, referencing all entities by their `@id` fields as per FAIR data principles.

### Dataset Source

The dataset source is defined by the Croissant schema, accessible at:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`


In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading

We load metadata and records from the dataset using `mlcroissant`. The dataset is defined by its Croissant schema URL.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')  # Suppress warnings for cleaner output

# Dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # Do not subscript or iterate, just use as an object
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Review all available record sets in the dataset, including their `@id` values, and then inspect their associated fields and columns using their own `@id`s.

In [ ]:
# List all record sets by @id
record_sets = dataset.record_sets  # List of mlc.RecordSet
print(f"Found {len(record_sets)} record sets:")
for rs in record_sets:
    print(f"  RecordSet @id: {rs.id_}, name: {getattr(rs, 'name', None)}")

# For each record set, list their fields and columns by @id
for rs in record_sets:
    print(f"\nRecordSet '@id': {rs.id_}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - Field @id: {field.id_}, name: {getattr(field, 'name', None)}, dataType: {getattr(field, 'data_type', None)}")
        if hasattr(field, 'columns') and field.columns:
            for col in field.columns:
                print(f"        - Column @id: {col.id_}, name: {getattr(col, 'name', None)}")

## 3. Data Extraction

We load data from each record set using their `@id` and extract the records into pandas DataFrames. All selection is done via the `@id` field. The columns of each DataFrame correspond to field `@id`s.

Replace `<record_set_id>` with an actual record set `@id` from the overview above to extract and examine records from that set.

In [ ]:
# Get all record set @ids
record_set_ids = [rs.id_ for rs in dataset.record_sets]
dataframes = dict()

for record_set_id in record_set_ids:
    # Each record_set_id is a string (e.g., 'cr:RecordSet/OrderedRegressionResults')
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded DataFrame for record set: {record_set_id}, shape: {df.shape}")

# For demonstration, use the first record set for EDA
if len(record_set_ids) > 0:
    main_record_set_id = record_set_ids[0]
    print(f"\nColumns in DataFrame for record set '{main_record_set_id}':")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

Let's perform some basic data processing. We'll select a numeric field (by its `@id`) to filter, normalize, and group. Update the field selection if needed based on the field listing above. All field references are by their `@id`.

In [ ]:
# Example: pick the first available record set
df = dataframes[main_record_set_id]
print(f"DataFrame columns (field @ids): {df.columns.tolist()}")

# Heuristic: find a likely numeric field to demonstrate
numeric_field_id = None
for col in df.columns:
    if df[col].dtype in ['int64', 'float64']:
        numeric_field_id = col
        break

if numeric_field_id is None:
    # Try to infer from column names
    for col in df.columns:
        if 'log' in col.lower() or 'coeff' in col.lower() or 'std' in col.lower() or 'se' in col.lower():
            if pd.to_numeric(df[col], errors='coerce').notnull().any():
                numeric_field_id = col
                break

if numeric_field_id:
    # Coerce to numeric for proper filtering
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].median()  # Use median as example threshold
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"\nFiltered records with {numeric_field_id} > {threshold} (using median):")
    display(filtered_df.head())
    
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, norm_col]].head())
    
    # Try grouping by a categorical column
    group_field = None
    for col in df.columns:
        if df[col].dtype == 'object' and col != numeric_field_id:
            group_field = col
            break
    
    if group_field:
        print(f"\nGrouped data by '{group_field}':")
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        display(grouped_df.head())
    else:
        print("No suitable categorical field found to group by.")
else:
    print("No numeric field detected in this record set for EDA.")

## 5. Visualization

Visualize distributions of the selected numeric field and its normalization, as well as grouped means if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and len(filtered_df) > 0:
    plt.figure(figsize=(10, 4))
    sns.histplot(filtered_df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id} (filtered records)")
    plt.xlabel(numeric_field_id)
    plt.show()

    plt.figure(figsize=(10, 4))
    norm_col = f"{numeric_field_id}_normalized"
    sns.histplot(filtered_df[norm_col].dropna(), bins=20, kde=True, color='green')
    plt.title(f"Distribution of Normalized {numeric_field_id} (filtered records)")
    plt.xlabel(norm_col)
    plt.show()

    if 'grouped_df' in locals() and not grouped_df.empty:
        plt.figure(figsize=(8,4))
        sns.barplot(x=group_field, y=numeric_field_id, data=grouped_df.head(10))
        plt.title(f"Mean {numeric_field_id} by {group_field} (top 10 categories)")
        plt.xticks(rotation=30)
        plt.show()
else:
    print("No numeric or grouped data to visualize.")

## 6. Conclusion

In this notebook, we demonstrated how to load and explore a FAIR dataset defined by a Croissant schema using `mlcroissant`. We enumerated record sets and fields by their `@id`, extracted data into DataFrames, filtered and normalized a numeric field, and visualized distributions. The approach ensures robust, metadata-driven exploration while keeping analysis reproducible and adaptable as the schema or dataset evolves.

**Further Directions:**
- Investigate other record sets or fields as needed, referencing their unique `@id`.
- Perform deeper statistical or modeling analyses per research objectives.
- Integrate with other FAIR^2 datasets or Croissant-recorded schemas for broader research.